# Signatures comparison — new sorted-cell held-out cohort (Figure 4)

The new sorted-cell cohort serves as a true held-out **test** cohort for the cell-type FGES benchmark, answering Reviewer 1's request to "Add new data of sorted cells to cell signature comparison" and the paper Methods commitment to ~75/25 train/test separation.

**Scope:** 16 of 20 FGES. The four rare-GOI FGES — `Main4_Th17_signature`, `Main4_Lymphatic_endothelium`, `Main4_Eosinophil_signature`, `Main4_Plasma_cells` — have too few held-out samples; in the supplementary heatmaps they are backfilled from cross-validated train-cohort scores.

**Random-FGES baseline:** v1 random gene lists are reused (loaded from `msigdb_gmt_paperS1.pkl`) but rescored on the new cohort so ranks stay comparable.

**Data:** input data is not distributed with this repository. Set the `<PATH_TO_...>` placeholders in the paths cell below before running. Outputs (pickles, SVGs, tables) go to `OUTPUT_DIR` with a `_new_cohort` suffix.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger

from signature_validation.benchmark.cohorts import (
    CONTROLS_ORDER,
    EXCLUDED_FGES_RARE,
    MAP_RAW,
    build_mapping,
    intersect_controls_with_cohort,
    load_new_cohort_annotation,
    load_new_cohort_expressions,
)
from signature_validation.benchmark.plotting import (
    plot_sens_spec_scatter,
    plot_signature_heatmap,
    plot_violin_per_source,
)
from signature_validation.benchmark.scoring import (
    compute_mapping_ssgseas,
    compute_out_table,
    fdr_correct_out,
)
from signature_validation.benchmark.signatures import (
    count_random_fges,
    harmonize_gmt_to_index,
    load_v1_msigdb_gmt,
    select_msigdb_gmt_subset,
)
from signature_validation.plotting.plotting import cells_p

sns.set_style("white")
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
# ── Data locations (the data are not distributed with the repository) ────
# Fill in the placeholders before running.
NEW_ANNOT_PATH = Path("<PATH_TO_NEW_COHORT_ANNOTATION_TSV>")          # annotation of the new (held-out) cohort
EXPR_PATH = "<PATH_TO_SORTED_CELL_EXPRESSIONS_DIR>"                  # expressions, RECOMPUTE=True only
TRAIN_SSGSEAS_PATH = Path("<PATH_TO_TRAIN_MAPPING_SSGSEAS_PKL>")      # mapping_ssgseas of the published (train) cohort
TRAIN_ANNOT_PATH = Path("<PATH_TO_TRAIN_CELLS_ANNOTATION_TSV>")       # train-cohort annotation (for S6.1)
CROSSVAL_SSGSEAS_PATH = "<PATH_TO_CROSSVAL_MAPPING_SSGSEAS_PKL>"      # cross-validated train scores (backfill)

OUTPUT_DIR = Path("../plots/new_cohort/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_SSGSEAS_PATH = OUTPUT_DIR / "mapping_ssgseas_new_cohort.pkl"
FGES_METRICS_PATH = OUTPUT_DIR / "fges_metrics_new_cohort.pkl"
OUT_TSV_PATH = OUTPUT_DIR / "out_new_cohort.tsv"
HEATMAP_PATH = OUTPUT_DIR / "signature_heatmap_new_cohort.svg"

# ── Run mode ──────────────────────────────────────────────────────────────
# RECOMPUTE=True  — recompute ssGSEA (mapping_ssgseas) and fges_metrics from scratch
#                   (slow; needs expressions in EXPR_PATH), then save the pickles.
# RECOMPUTE=False — load the cached pickles (MAPPING_SSGSEAS_PATH /
#                   FGES_METRICS_PATH) and only build plots/tables
#                   (expressions are not loaded).
RECOMPUTE = True

logger.info("RECOMPUTE={}", RECOMPUTE)
logger.info("new annotation:    {}", NEW_ANNOT_PATH)
logger.info("mapping_ssgseas:   {}", MAPPING_SSGSEAS_PATH)

In [ ]:
public_cells_annot = load_new_cohort_annotation(NEW_ANNOT_PATH)
public_cells_annot["Cell_type"].value_counts()

In [ ]:
# ── Held-out test: drop samples already present in the published cohort ──
# 65 of the 30,411 new-cohort samples also appear in the v1 pickle (train). While they
# stay, Figure 4 partly measures the same data the signatures were selected on.
# The removal is concentrated: exactly two types are affected — Mast_cells 58 → 8
# (drops below min_n and goes to the merge backfill, see the supplementary-figure
# data-prep cell) and Follicular_T_helpers 265 → 250 (no consequences).
from signature_validation.benchmark.cohorts import (
    drop_train_samples,
    load_train_sample_ids,
)

train_ids = load_train_sample_ids(TRAIN_SSGSEAS_PATH)
public_cells_annot, train_drop_report = drop_train_samples(public_cells_annot, train_ids)
train_drop_report

In [ ]:
from signature_validation.utils.utils import read_expressions

In [ ]:
# Expressions are needed only for recomputation (ssGSEA + metrics).
if RECOMPUTE:
    public_cells_expr = read_expressions(public_cells_annot, path=EXPR_PATH)
    logger.info("expressions: {}", public_cells_expr.shape)
else:
    public_cells_expr = None
    logger.info("RECOMPUTE=False — skipping expression loading")

In [ ]:
V1_GMT_PICKLE = "./data/msigdb_gmt_paperS1.pkl"
v1_gmt_full = load_v1_msigdb_gmt(V1_GMT_PICKLE)
in_scope_fges = [k for k in MAP_RAW if k not in EXCLUDED_FGES_RARE]
v1_gmt = select_msigdb_gmt_subset(v1_gmt_full, in_scope_fges)

for sign in in_scope_fges:
    assert sign in v1_gmt[sign], f"v1 GMT[{sign}] missing the BG sub-signature"
    n_random = count_random_fges(v1_gmt[sign])
    assert n_random == 10, f"{sign}: expected 10 RANDOM_FGES, got {n_random}"

if RECOMPUTE:
    msigdb_gmt = harmonize_gmt_to_index(v1_gmt, public_cells_expr.index)
else:
    # Without recomputation only sub-signature NAMES (not genes) are needed — use v1_gmt as is.
    msigdb_gmt = v1_gmt
logger.info(
    "msigdb_gmt: {} FGES, {} signatures total",
    len(msigdb_gmt),
    sum(len(v) for v in msigdb_gmt.values()),
)

In [ ]:
mapping = build_mapping(annotation=public_cells_annot)
controls_present = intersect_controls_with_cohort(CONTROLS_ORDER, public_cells_annot)
logger.info(
    "in-scope: {} FGES; controls present in new cohort: {}",
    len(mapping),
    len(controls_present),
)
for sign, bucket in mapping.items():
    logger.info(
        "{}: GOI={}, Control={}, Deleted={}",
        sign,
        bucket["Goi"],
        len(bucket["Control"]),
        len(bucket["Deleted_controls"]),
    )

In [ ]:
from signature_validation.benchmark.cohorts import PARENT_TO_DAUGHTER
from signature_validation.benchmark.scoring import clean_parent_daughter_goi

if RECOMPUTE:
    mapping_ssgseas = compute_mapping_ssgseas(
        public_cells_expr=public_cells_expr,
        public_cells_annot=public_cells_annot,
        mapping=mapping,
        msigdb_gmt=msigdb_gmt,
    )
    # parent→daughter cleanup: without it the scores of
    # Macrophages/Monocytes are distorted (shared signatures stay in the parent's GOI).
    mapping_ssgseas = clean_parent_daughter_goi(mapping_ssgseas, PARENT_TO_DAUGHTER)
    with open(MAPPING_SSGSEAS_PATH, "wb") as fh:
        pickle.dump(mapping_ssgseas, fh, pickle.HIGHEST_PROTOCOL)
    logger.info("computed+cleaned mapping_ssgseas → {}", MAPPING_SSGSEAS_PATH)
else:
    with open(MAPPING_SSGSEAS_PATH, "rb") as fh:
        mapping_ssgseas = pickle.load(fh)
    logger.info("loaded mapping_ssgseas from {}", MAPPING_SSGSEAS_PATH)

for sign in mapping_ssgseas:
    if sign in EXCLUDED_FGES_RARE:
        continue
    assert mapping_ssgseas[sign]["Goi"], f"{sign}: GOI cohort is empty"

## Score correctness, metrics, scatters and Supplement tables

`clean_parent_daughter_goi` reproduces the v1 `parent_to_daughter` cleaning (Scater_plots cell 16) that the first draft omitted — without it macrophage/monocyte GOI frames keep daughter-shared signatures and the heatmap / `out` table / scatter are wrong. Then `fges_metrics` (bootstrap F1/AUC + rank CV), the F1-vs-CV source scatters (Fig 4 F/G) and the S4/S6 Supplement tables.

In [ ]:
# Sample-count check for macrophages — recompute mode only
# (needs expressions): annotation vs. expressions vs. GOI frames.
if RECOMPUTE:
    MACRO_GOI = {
        "Main4_Pan_macrophage_signature": "Macrophages",
        "Main4_M2_signature": "Macrophages_M2",
        "Main4_Monocyte": "Monocytes",
    }

    expr_cols = set(public_cells_expr.columns)
    ct_counts = public_cells_annot["Cell_type"].value_counts()

    for fges, ct in MACRO_GOI.items():
        n_annot = int(ct_counts.get(ct, 0))
        samples_of_ct = public_cells_annot.index[public_cells_annot["Cell_type"] == ct]
        n_expr = len(expr_cols.intersection(samples_of_ct))
        goi_frames = mapping_ssgseas.get(fges, {}).get("Goi", {})
        n_goi = int(goi_frames[ct].shape[0]) if ct in goi_frames else 0
        dropped = n_annot - n_expr
        logger.info(
            "{ct:16s} | FGES={fges:34s} | annot={n_annot:5d} | with_expr={n_expr:5d} "
            "| GOI_frame_rows={n_goi:5d} | dropped_no_expr={dropped:5d}",
            ct=ct, fges=fges, n_annot=n_annot, n_expr=n_expr, n_goi=n_goi, dropped=dropped,
        )
        if n_goi != n_expr:
            logger.warning(
                "{ct}: GOI frame ({n_goi}) != samples with expressions ({n_expr}) — "
                "check for duplicate indices / filtering",
                ct=ct, n_goi=n_goi, n_expr=n_expr,
            )

    all_cts = {ct for b in mapping.values() for g in ("Goi", "Control", "Deleted_controls") for ct in b[g]}
    total_samples = public_cells_annot.index[public_cells_annot["Cell_type"].isin(all_cts)]
    total_with_expr = len(expr_cols.intersection(total_samples))
    logger.info("TOTAL samples (all in-scope cell types, with expressions): {}", total_with_expr)
else:
    logger.info("RECOMPUTE=False — skipping the macrophage sample-count check")

In [ ]:
out = compute_out_table(mapping_ssgseas, mapping, msigdb_gmt, controls_present)
out = fdr_correct_out(out, controls_present)
out.to_csv(OUT_TSV_PATH, sep="\t")
logger.info("wrote {} ({} rows × {} cols)", OUT_TSV_PATH, *out.shape)
out.head()

In [ ]:
# plot_violin_per_source(mapping_ssgseas, save_dir=OUTPUT_DIR)
plot_signature_heatmap(
    mapping_ssgseas=mapping_ssgseas,
    out_df=out,
    mapping=mapping,
    msigdb_gmt=msigdb_gmt,
    annotation=public_cells_annot,
    controls_order=controls_present,
    palette={ct: cells_p[ct] for ct in controls_present if ct in cells_p},
    save_path=HEATMAP_PATH,
    short=True,
)
averaged = plot_sens_spec_scatter(
    mapping_ssgseas=mapping_ssgseas,
    msigdb_gmt=msigdb_gmt,
    mapping=mapping,
    save_dir=OUTPUT_DIR,
)
logger.info("plots saved under {}", OUTPUT_DIR)

In [ ]:
msigdb_gmt['Main4_Pan_macrophage_signature']['Main4_Pan_macrophage_signature'].genes

In [ ]:
# fges_metrics: bootstrap classification (F1/Accuracy/PR-AUC/ROC-AUC) + rank CV
# Recompute, or load the cached pickle.
from signature_validation.benchmark.metrics import compute_fges_metrics

# get_strat_cell_type takes min_samples = the MINIMUM over control types, so
# one tiny control truncates every subsample. The reference analysis
# dropped only Th2_cells for this, but in the new cohort the bottleneck
# is wider: Monocytic_DC=1, Plasmablasts=3, Th2_cells=8, Th1_cells=8 scored
# samples → min_samples=1 and the control would collapse to ~37 samples per iteration.
# Drop all controls below the threshold; GOI cohorts are unaffected — they come
# from mapping_ssgseas["Goi"], and public_cells_annot is only used to stratify
# controls (i.e. Th1_cells remains a full GOI for Main4_Th1_signature).
MIN_CONTROL_SAMPLES = 10

if RECOMPUTE:
    ranked_expr = public_cells_expr.rank(pct=True)
    pipeline_genes = public_cells_expr.index.to_list()

    control_samples = pd.Index(
        sorted(
            {
                sample
                for bucket in mapping_ssgseas.values()
                for df in bucket.get("Control", {}).values()
                for sample in df.index
            }
        )
    )
    control_counts = (
        public_cells_annot["Cell_type"].reindex(control_samples).dropna().value_counts()
    )
    tiny_controls = sorted(control_counts.index[control_counts < MIN_CONTROL_SAMPLES])
    logger.info(
        "dropping controls < {} samples: {} | min_samples becomes {}",
        MIN_CONTROL_SAMPLES,
        {ct: int(control_counts[ct]) for ct in tiny_controls},
        int(control_counts[control_counts >= MIN_CONTROL_SAMPLES].min()),
    )
    annot_for_metrics = public_cells_annot[
        ~public_cells_annot["Cell_type"].isin(tiny_controls)
    ]
    

    fges_metrics = compute_fges_metrics(
        mapping_ssgseas=mapping_ssgseas,
        msigdb_gmt=msigdb_gmt,
        public_cells_annot=annot_for_metrics,
        ranked_expr=ranked_expr,
        pipeline_genes=pipeline_genes,
        n_iter=10,
    )
    with open(FGES_METRICS_PATH, "wb") as fh:
        pickle.dump(fges_metrics, fh, pickle.HIGHEST_PROTOCOL)
    logger.info("computed fges_metrics → {} ({} FGES cols)", FGES_METRICS_PATH, len(fges_metrics))
else:
    with open(FGES_METRICS_PATH, "rb") as fh:
        fges_metrics = pickle.load(fh)
    logger.info("loaded fges_metrics from {} ({} FGES cols)", FGES_METRICS_PATH, len(fges_metrics))

In [ ]:
# F1 vs rank-CV scatter plots by FGES source (as in Figure 4 panels F/G).
# Written to OUTPUT_DIR so the published paper SVGs are NOT overwritten.
from signature_validation.benchmark.metrics import plot_f1_cv_scatters

plot_f1_cv_scatters(
    fges_metrics=fges_metrics,
    msigdb_gmt=msigdb_gmt,
    save_dir=OUTPUT_DIR,
)
logger.info("F1/CV scatters → {}", OUTPUT_DIR / "svg_pictures_F1_cv")

In [ ]:
# Supplementary tables: S4.x (FGES performance per cell type, e.g. B_cells)
# + S6.1 (dataset inventory: how many samples of each dataset actually reached
# the figures and how they split into train / holdout).
import os

from signature_validation.benchmark.cohorts import (
    collect_sample_ids,
    load_train_annotation,
)
from signature_validation.benchmark.tables import (
    build_dataset_list_table,
    build_fges_performance_tables,
)

TABLES_DIR = (OUTPUT_DIR / "tables").resolve()
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# detect_fges_source reads ./data/msigdb...gmt relative to cwd — temporarily
# move one level up (Cell_type_FGES_comparison), where that folder exists.
_cwd0 = os.getcwd()
try:
    os.chdir("..")
    s4_tables = build_fges_performance_tables(
        mapping_ssgseas=mapping_ssgseas,
        fges_metrics=fges_metrics,
        mapping=mapping,
        msigdb_gmt=msigdb_gmt,
        save_dir=TABLES_DIR,
        prefix="S4",
    )
finally:
    os.chdir(_cwd0)

# ── S6.1: Sorted cell dataset list ───────────────────────────────────────
# N counts SCORED samples (union of Goi + Control + Deleted_controls),
# not annotation rows: only a sample that passed QC and had expressions
# has a score, i.e. exactly what the figures show. Counting annotation rows would give
# 30,411 samples instead of 16,976 — the table would describe a different experiment.
#
# Train = the whole published v1 cohort (7,990 samples): the signatures were selected
# on it, and the cross-validation backfill takes rare cell types from it
# (Th17 / Eosinophils / Endothelium_lymph) for the supplementary figures.
#
# fallback_annotation is re-read instead of taken from public_cells_annot: 44
# train samples (tonsil Tfh and mast cells, added to v1 manually)
# are missing from the v1 annotation and exist only in the new one — but
# drop_train_samples removed them, so the post-drop variable no longer has them.
holdout_ids = collect_sample_ids(mapping_ssgseas)
train_annot = load_train_annotation(
    train_ids,
    path=TRAIN_ANNOT_PATH,
    fallback_annotation=load_new_cohort_annotation(NEW_ANNOT_PATH),
)

s6_table = build_dataset_list_table(
    public_cells_annot,
    TABLES_DIR / "S6.1_sorted_cell_datasets.tsv",
    holdout_samples=holdout_ids,
    train_samples=train_ids,
    train_annotation=train_annot,
)

assert s6_table["N_holdout"].sum() == len(holdout_ids)
assert s6_table["N_train"].sum() == len(train_ids)
assert (s6_table["N_samples"] == s6_table["N_train"] + s6_table["N_holdout"]).all()
assert s6_table["Dataset"].notna().all()
assert not s6_table["Dataset"].duplicated().any()

_both = int(((s6_table["N_train"] > 0) & (s6_table["N_holdout"] > 0)).sum())
logger.info(
    "wrote {} S4 tables + S6.1: {} datasets | train {} | holdout {} | "
    "in both cohorts {} → {}",
    len(s4_tables),
    len(s6_table),
    int(s6_table["N_train"].sum()),
    int(s6_table["N_holdout"].sum()),
    _both,
    TABLES_DIR,
)
s6_table.head()

### Table S6.1 — what to state in the caption

**Caption (paper-ready).** *Table S6.1. Sorted cell dataset list.* Datasets contributing
samples to the cell-type FGES benchmark. `N_samples` counts only samples with a computed
ssGSEA score — i.e. samples that passed technical QC, carried expression data and belong to
one of the benchmarked cell types — split into the published training cohort (`N_train`,
7,990 samples) and the held-out sorted-cell cohort (`N_holdout`, 16,976 samples).
`Cell_types` lists the benchmarked cell types the dataset contributes, semicolon-separated.

**Caveats worth keeping in the Methods text:**

- **Denominator.** 1,300 datasets / 24,966 samples is the level of *scored* samples.
  Counting annotation rows gives 1,259 (new) + 736 (old) datasets and 30,411
  samples in the holdout alone: a different number with a different meaning. Any "shared studies"
  figure must be quoted together with its denominator.
- **The split is at the sample level, not the study level.** 128 datasets contribute samples to both cohorts
  (468 train only, 704 holdout only). The sample overlap is exactly zero —
  `drop_train_samples` removes the 65 shared IDs. So the holdout is truly held out
  by sample, but not by batch.
- **Rare cell types come from train.** `Th17_cells`, `Eosinophils`, `Endothelium_lymph`
  have no holdout samples at all, and `Th2_cells` / `Mast_cells` / `Monocytic_DC` have
  fewer than 10; in the supplementary figures they are backfilled from train-cohort cross-validation
  (`backfill_rare_cell_types`, `min_n=10`, `mode="merge"`). These samples are already counted
  in the `N_train` column.
- **Labels are harmonized.** Tonsillar Tfh from train (`Follicular_T_helper_tonsil`) is mapped
  to `Follicular_T_helpers`, otherwise one population would be listed twice. Biological
  equivalence of the two definitions does not follow from the annotation — if a reviewer cares,
  check the sorting markers.
- **Internal dataset** — one dataset in the table is internal and has no public accession.

## Supplementary figures — ssGSEA medians by cell type

Supplementary figures built on the same `mapping_ssgseas`, all on **medians of raw (unscaled) ssGSEA** per cell type.

**X axis (all figures), strictly in this order:** T cells, CD4 T cells, Th1 cells, Th2 cells, Th17 cells, Follicular T helpers, Tregs, CD8 T cells, NK cells, B cells, Plasma B cells, Plasmablasts, Neutrophils, Eosinophils, Mast cells, Monocytes, Macrophages, Macrophages M1, Macrophages M2, Endothelium, Endothelium lymph, **Other controls** — all other scored types are pooled into the last column.

**Styling (shared by both sets).** Cells are **square**; there are **no** numbers inside cells — the colorbar carries their value. Color encodes the **raw ssGSEA median** (not a z-score), so the colorbar is labeled in ssGSEA units and reads directly. Color limits come from the **full** matrix of all FGES and are only then sliced per file; otherwise colors of two files would not be comparable.

The color scale is linear. `color_scale="symlog"` is available (signed log10: linear inside ±`linthresh`, log10 outside — plain log10 does not work, ~8% of medians are negative) but is off by default: the medians span less than one decade (99.5% of cells in 10³–10⁴), so log10 squeezes the useful contrast (Macrophages 8688 vs NK 2955 → 3.94 vs 3.47) into one twelfth of the bar and spends the rest on the nearly empty region around zero.

**Set 1 — one SVG per FGES, for each selection method.** Written to `heatmaps_per_fges/<selection>/<FGES>.svg` plus one `_colorbar.svg` per folder.

The main selection is `separation`, "**highest in GOI, lowest in control**". Rows are ranked by the **median gap**:

```
gap = median(GOI) − median(Control)
```

Ranking by `gap` is used because it is exactly the separation the reader sees in the figure: cells are colored by raw medians, so the widest gap is the widest color jump. Cohen's d `(mean(GOI) − mean(Control)) / pooled_sd` is computed alongside and reported in the table, but does not drive the sort: it divides by the pooled SD and therefore penalizes a signature for a wide spread within GOI that the figure does not show. The flip side of `gap`: sub-signatures of one FGES live on different ssGSEA scales, so a gene set that is large in absolute terms can win on gap without separating the cohorts better in relative units — that is what the `d` column in the table is for.

It is computed directly on the ssGSEA the figure draws, not on `fges_metrics`. Two consequences follow: the cross-validation backfill is included (`fges_metrics` does not know about it), and the same MSigDb set under two FGES is scored separately under each — so the `ambiguous_metric` mode, where a set inherited the GOI/control split of another block, disappears. `Deleted_controls` are **not** part of the control: these are types the FGES excludes as biologically overlapping (`CONTROLS_TO_DELETE`; for pan-macrophage — Monocytes and M1/M2), and counting them as control would penalize the signature for working correctly. Under this selection `Main4_Pan_macrophage_signature` ranks **1st of 261** (gap = 10,031 with GOI 7866 vs control −2166). For comparison, the previous rank composite `rank(metric) + rank(−goi_cv)` placed it 15th–19th and kept it out of the top 10 on all seven metrics.

The previous selection by bootstrap metrics (`F1`, `Accuracy`, `Precision_score`, `Recall_score`, `Average_precision`, `ROC_AUC`, `PR_AUC` in a composite with `goi_cv`) is kept and writes its own seven folders — with the same new styling.

**Set 2 — internal specificity.** Rows: internal (BG) signatures only, all 19. Color is the same raw ssGSEA median on the shared scale.

**Cross-validation backfill, `mode="merge"`.** Types with zero test samples (`Th17_cells`, `Eosinophils`, `Endothelium_lymph`) come entirely from cross-validation of the original cohort — there is nothing to merge with. Types that have test samples but fewer than 10 (`Th2_cells` 8, `Mast_cells` 8 after removing train samples, `Monocytic_DC` 1) **merge test and train** and are scored on all samples at once. The threshold of 10 is passed to `backfill_rare_cell_types` explicitly; the shared `DEFAULT_MIN_N = 20` in `crossval.py` is unchanged. At threshold 10, `Plasmablasts` (12), `Keratinocytes` (15) and `Fibroblast_line` (10) are no longer backfilled from train and stay purely held out. Markers: `*` — the whole column comes from cross-validation, `†` — the whole signature does, `‡` — the column/row contains merged test+train data.

Merging at the score level is valid because ssGSEA is a single-sample method: a sample's score does not depend on cohort composition. Columns are intersected; a gene-set version mismatch is logged rather than silently corrupting the medians.

**Row order** is set by `fges_order_by_cell_type` — FGES are sorted by the position of their GOI type on the X axis.

In [ ]:
# ── Supplementary figures: data preparation ──────────────────────────────
# Rare cell types of the new cohort (Th17_cells / Eosinophils / Endothelium_lymph
# = 0 samples; Th2_cells = 8; Mast_cells = 8 after removing train;
# Monocytic_DC = 1) are backfilled with cross-validated scores of the original cohort.
#
# The threshold here is min_n=10, passed explicitly: DEFAULT_MIN_N=20 in crossval.py is shared
# by all notebooks and stays as is. At threshold 10 these are NOT backfilled from train and
# stay purely held out: Plasmablasts (12), Keratinocytes (15) and
# Fibroblast_line (10) — at threshold 20 they were merged.
#
# mode="merge": where test samples EXIST but are few, test and train are
# MERGED and scored on all samples at once (provenance "merged", marked ‡ in
# the figures). Where there are no test samples at all, there is nothing to merge
# with — such types stay purely cross-validated ("cross_validation", †/*).
# It also adds the 4 rare FGES that are not in this notebook's 15-FGES scope.
from signature_validation.benchmark.cohorts import MAP_RAW_RARE
from signature_validation.benchmark.crossval import (
    backfill_rare_cell_types,
    load_crossval_ssgseas,
)
from signature_validation.benchmark.heatmaps_supp import (
    CV_METRIC_KEY,
    HIGHER_IS_BETTER_METRICS,
    OTHER_CONTROLS_MEMBERS,
    SEPARATION_KEY,
    SUPP_COLUMN_ORDER,
    fges_order_by_cell_type,
    plot_internal_specificity_heatmap,
    plot_top_metric_heatmaps,
)

crossval_ssgseas = load_crossval_ssgseas(CROSSVAL_SSGSEAS_PATH)
mapping_ssgseas_supp, provenance_supp = backfill_rare_cell_types(
    mapping_ssgseas, crossval_ssgseas, public_cells_annot, min_n=10, mode="merge"
)

# FGES are sorted by the position of their GOI type on the X axis → the diagonal reads.
FGES_ORDER_15 = fges_order_by_cell_type(MAP_RAW)                      # have metrics
FGES_ORDER_19 = fges_order_by_cell_type({**MAP_RAW, **MAP_RAW_RARE})  # + 4 rare

# Provenance breakdown: how many (FGES × group × cell_type) come from which source.
prov_counts = pd.Series(
    [
        label
        for groups in provenance_supp.values()
        for labels in groups.values()
        for label in labels.values()
    ]
).value_counts()
logger.info("provenance (FGES×group×cell_type): {}", prov_counts.to_dict())
logger.info(
    "supp heatmaps: {} FGES after backfill; {} named columns + Other controls ({} pooled types)",
    len(mapping_ssgseas_supp),
    len(SUPP_COLUMN_ORDER),
    len(OTHER_CONTROLS_MEMBERS),
)

In [ ]:
# ── Figures: ONE SVG per FGES, a separate folder per selection method ────
# Main selection — SEPARATION_KEY: "highest in GOI, lowest in
# control". Rows are ranked by the MEDIAN GAP between GOI samples and the POOLED
# controls of the same FGES: gap = median(GOI) − median(Control). gap is used because
# it is the separation visible in the figure — cells are colored by raw
# medians, so the widest gap is the widest color jump.
# Cohen's d is computed alongside and stored in the table as column d, but does not sort: it
# divides by the pooled SD and penalizes a signature for spread within GOI that the
# figure does not show. The flip side of gap: sub-signatures of one FGES live on
# different ssGSEA scales, so a large gene set can win on gap without
# separating the cohorts better in relative units — hence the d column.
#
# Computed directly on the ssGSEA the figure draws, so (a) the cross-validation
# backfill is included (fges_metrics does not know about it), (b) the same
# MSigDb set under two FGES is scored separately under each.
#
# Deleted_controls are NOT part of the control: these are types the FGES excludes as
# biologically overlapping (for pan-macrophage — Monocytes and M1/M2), and
# counting them as control would penalize the signature for working correctly.
#
# Color is the RAW ssGSEA median (no z-score); limits come from the FULL matrix
# of all FGES and are only then sliced per file: otherwise colors of two files
# are not comparable. No numbers in cells — the colorbar carries their value. The scale
# is linear; color_scale="symlog" is available but off by default: the medians
# span less than one decade (99.5% of cells in 10^3–10^4), and log10 would squeeze
# the useful contrast (Macrophages 8688 vs NK 2955 → 3.94 vs 3.47) into 1/12 of the bar.
PER_FGES_DIR = OUTPUT_DIR / "heatmaps_per_fges"

medians_sep, ranking_sep = plot_top_metric_heatmaps(
    mapping_ssgseas=mapping_ssgseas_supp,
    fges_metrics=fges_metrics,  # unused for the separation selection
    msigdb_gmt=msigdb_gmt,
    fges_order=FGES_ORDER_15,
    save_dir=PER_FGES_DIR,
    metric=SEPARATION_KEY,
    top_k=10,
    provenance=provenance_supp,
)
ranking_sep.to_csv(
    TABLES_DIR / "top10_separation_ranking_new_cohort.tsv", sep="\t", index=False
)
medians_sep.to_csv(TABLES_DIR / "top10_separation_medians_new_cohort.tsv", sep="\t")

# How many internal (BG) signatures land in their own top 10 — a direct check
# that the selection lifts the FGES's "own" signature to the top.
internal_sep = ranking_sep[ranking_sep["is_internal"]]
logger.info(
    "separation: internal signatures in their own top 10 — {}/{}",
    len(internal_sep),
    len(FGES_ORDER_15),
)

# The previous bootstrap-metric selection is kept for the supplement: the same seven folders,
# now with square cells, no annotations and raw medians as color.
metric_rankings = {}
for metric in HIGHER_IS_BETTER_METRICS:
    medians_m, ranking_m = plot_top_metric_heatmaps(
        mapping_ssgseas=mapping_ssgseas_supp,
        fges_metrics=fges_metrics,
        msigdb_gmt=msigdb_gmt,
        fges_order=FGES_ORDER_15,
        save_dir=PER_FGES_DIR,
        metric=metric,
        cv_key=CV_METRIC_KEY,
        top_k=10,
        provenance=provenance_supp,
    )
    ranking_m.to_csv(
        TABLES_DIR / f"top10_{metric}_ranking_new_cohort.tsv", sep="\t", index=False
    )
    medians_m.to_csv(TABLES_DIR / f"top10_{metric}_medians_new_cohort.tsv", sep="\t")
    metric_rankings[metric] = ranking_m

logger.info(
    "done: separation + {} metrics × {} FGES → {}",
    len(metric_rankings),
    ranking_sep["FGES"].nunique(),
    PER_FGES_DIR,
)
internal_sep[
    ["FGES", "Rank_in_fges", "Pool", "gap", "d", "goi_median", "control_median"]
]

In [ ]:
# ── Figure 2: specificity of internal (BG) signatures ────────────────────
# Shows across which populations the UNscaled ssGSEA of a single
# signature diverges. Color is the raw median on the shared scale; cell numbers are removed:
# their value reads from the colorbar.
INTERNAL_HEATMAP_PATH = (
    OUTPUT_DIR / "heatmap_internal_specificity_raw_median_new_cohort.svg"
)

medians_internal = plot_internal_specificity_heatmap(
    mapping_ssgseas=mapping_ssgseas_supp,
    fges_order=FGES_ORDER_19,
    save_path=INTERNAL_HEATMAP_PATH,
    colorbar_path=INTERNAL_HEATMAP_PATH.with_name(
        INTERNAL_HEATMAP_PATH.stem + "_colorbar.svg"
    ),
    provenance=provenance_supp,
)

medians_internal.to_csv(
    TABLES_DIR / "internal_specificity_medians_new_cohort.tsv", sep="\t"
)
medians_internal.round(0)